In [68]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def weighted_kmeans_clustering(df, k=10, location_vars=None, weights=None, size_var=None, id_col=None, random_state=42, max_iterations=100, verbose=None, init_centroids=None):
    """
    Desc: weighted K-means clustering to nearest centroid.
    
    ==========
    Parameters
    ==========
    df              : The dataframe containing data
    k               : Number of clusters to create
    location_vars   : List of column to use as location variables (default: all numeric columns)
    weights         : Variable used as weights (default: equal weights)
    size_var        : Column name to use for size weighting (default: None = equal weights of 1)
    id_col          : Column name for observation identification (default: None = use dataframe index)
    random_state    : Random seed
    max_iterations  : Maximum iterations
    verbose         : None (no output), 1 (summary only), 2 (full details)
    init_centroids  : data points (ex: [0, 5, 10]) or None for random initialization

    ==========
    Returns
    ==========
    cluster representatives dataframe
    """

    df = df.copy()

    # default size variable (all equal weights)
    if size_var is None:
        df['_temp_size_weight'] = 1.0
        size_var = '_temp_size_weight'
    
    # default id column (use index)
    if id_col is None:
        df['_temp_id'] = df.index
        id_col = '_temp_id'

    # determine which columns to use for clustering
    if location_vars is None:
        location_vars = df.select_dtypes(include=[np.number]).columns.tolist()
        # Remove size and id columns if they're in the numeric columns
        for col in [size_var, id_col, '_temp_size_weight', '_temp_id']:
            if col in location_vars:
                location_vars.remove(col)
            
    # set default weights (all features equally important)
    if weights is None:
        weights = {var: 1.0 for var in location_vars}
    
    # extract and standardize location data
    X_raw = df[location_vars].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    X_scaled_df = pd.DataFrame(X_scaled, columns=location_vars)
    
    # apply weights
    for var in location_vars:
        if var in weights:
            X_scaled_df[var] = X_scaled_df[var] * weights[var]
    
    X = X_scaled_df.values
    n_observations = len(X)

    # observation sizes and normalize weights
    size_weights = df[size_var].values
    normalized_weights = size_weights / size_weights.sum() * len(size_weights)
    

    # Initialize centroids
    if init_centroids is None:
        # random initialization
        np.random.seed(random_state)
        indices = np.random.choice(n_observations, k, replace=False)
        centroids = X[indices]
        init_method = "random"
    elif isinstance(init_centroids, list) and len(init_centroids) == k:
        # data points initialization
        indices = np.array(init_centroids)
        if np.any(indices >= n_observations) or np.any(indices < 0):
            raise ValueError(f"Initial centroid indices must be between 0 and {n_observations-1}")
        centroids = X[indices]
        init_method = "indices"
    else:
        raise ValueError(f"init_centroids must be None (random initialization) or a list of indices (ex: [0, 5, 10]) of length k={k}")
    
    if verbose is not None:
        print("=" * 50)
        print(f"INITIALIZATION")
        print("=" * 50)
        print(f"Number of clusters (k): {k}")
        print(f"Number of observations: {n_observations}")
        print(f"Number of features: {len(location_vars)}")
        print(f"Feature names: {location_vars}")
        print(f"Feature weights: {weights}")
        print(f"Initialization method: {init_method}")
        print(f"Random seed: {random_state}")
        print(f"Initial centroid indices: {indices}")
        if verbose >= 2:
            print(f"\nInitial centroid:")
            for i, centroid in enumerate(centroids):
                print(f"  Cluster {i}: {centroid}")
        print()
    
    # K-means iterations
    for iteration in range(max_iterations):
        # Calculate distance from each observation to each centroid
        distances = np.zeros((n_observations, k))
        
        for centroid_idx in range(k):
            centroid_position = centroids[centroid_idx]
            
            # calculate distance from each observation to this centroid
            for obs_idx in range(n_observations):
                obs_position = X[obs_idx]
                squared_differences = (obs_position - centroid_position) ** 2
                distance = np.sqrt(squared_differences.sum())
                distances[obs_idx, centroid_idx] = distance
        
        # assign each observation to nearest centroid
        labels = np.argmin(distances, axis=1)
        
        # Update centroids using weighted average
        new_centroids = np.zeros_like(centroids)
        
        for cluster_id in range(k):
            observations_in_cluster = (labels == cluster_id)
            
            if np.sum(observations_in_cluster) > 0:
                # calculate weighted average position of all observations in cluster
                cluster_points = X[observations_in_cluster]
                cluster_weights = normalized_weights[observations_in_cluster]
                new_centroids[cluster_id] = np.average(cluster_points, axis=0, weights=cluster_weights)
            else:
                # keep old centroid if cluster is empty
                new_centroids[cluster_id] = centroids[cluster_id]
        
        # convergence
        if verbose is not None:
            cluster_sizes = np.bincount(labels, minlength=k)
            print(f"Iteration {iteration + 1}:")
            print(f"  Cluster sizes: {cluster_sizes}")
            if verbose >= 2:
                print(f"New centroid positions:")
                for i, centroid in enumerate(new_centroids):
                    print(f"  Cluster {i}: {centroid}")
            print()
        
        if np.array_equal(new_centroids, centroids):
            if verbose is not None:
                print(f"Converged after {iteration + 1} iterations")
                print("=" * 50)
            break
        
        centroids = new_centroids
    else:
        if verbose is not None:
            print(f"Not converged after maximum iteration of = ({max_iterations})")
            print("=" * 50)
    
# find representative seriatim (nearest to centroid)
    df['cluster'] = labels
    representatives = []
    
    for cluster_id in range(k):
        observations_in_cluster = (labels == cluster_id)
        
        if np.sum(observations_in_cluster) > 0:
            # calculate distances from all points in a cluster to its centroid
            cluster_distances = np.linalg.norm(X[observations_in_cluster] - centroids[cluster_id], axis=1)
            
            # find observation with minimum distance
            closest_idx = np.where(observations_in_cluster)[0][np.argmin(cluster_distances)]
            cluster_weight_sum = df.loc[observations_in_cluster, size_var].sum()
            
            rep_info = {
                'cluster': cluster_id,
                'id': df.iloc[closest_idx][id_col],
                'cluster_size': np.sum(observations_in_cluster),
                'cluster_weight_sum': cluster_weight_sum
            }
            representatives.append(rep_info)
    
    return pd.DataFrame(representatives)

In [72]:
df = pd.read_excel('./SHEETS/k-means testing.xlsx', sheet_name='data_sample')
df = df.drop(columns=['No', 'cluster'])

result = weighted_kmeans_clustering(
    df=df,
    k=3,
    location_vars=['var_1', 'var_3'],
    weights={'var_1': 1, 'var_3': 1},
    random_state=42,
    verbose=2,
    init_centroids=[7, 8, 9]
)

print(result)


INITIALIZATION
Number of clusters (k): 3
Number of observations: 10
Number of features: 2
Feature names: ['var_1', 'var_3']
Feature weights: {'var_1': 1, 'var_3': 1}
Initialization method: indices
Random seed: 42
Initial centroid indices: [7 8 9]

Initial centroid:
  Cluster 0: [-0.41904997  1.35710229]
  Cluster 1: [-0.19253647 -1.1874645 ]
  Cluster 2: [-0.36242159 -1.61155897]

Iteration 1:
  Cluster sizes: [4 5 1]
New centroid positions:
  Cluster 0: [-0.68803474  0.93300782]
  Cluster 1: [ 0.62291211 -0.42409446]
  Cluster 2: [-0.36242159 -1.61155897]

Iteration 2:
  Cluster sizes: [5 3 2]
New centroid positions:
  Cluster 0: [-0.73616886  0.67855114]
  Cluster 1: [ 1.41193412 -0.19791075]
  Cluster 2: [-0.27747903 -1.39951173]

Iteration 3:
  Cluster sizes: [5 3 2]
New centroid positions:
  Cluster 0: [-0.73616886  0.67855114]
  Cluster 1: [ 1.41193412 -0.19791075]
  Cluster 2: [-0.27747903 -1.39951173]

Converged after 3 iterations
   cluster   id  cluster_size  cluster_weight_s

In [71]:
df = pd.read_excel('./SHEETS/k-means testing.xlsx', sheet_name='data_sample')
df = df.drop(columns=['No', 'cluster'])

result = weighted_kmeans_clustering(
    df=df,
    k=3,
    location_vars=['var_1', 'var_3'],
    weights={'var_1': 1, 'var_3': 3},
    random_state=42,
    verbose=2,
    init_centroids=[1, 5, 9]
)

print(result)


INITIALIZATION
Number of clusters (k): 3
Number of observations: 10
Number of features: 2
Feature names: ['var_1', 'var_3']
Feature weights: {'var_1': 1, 'var_3': 3}
Initialization method: indices
Random seed: 42
Initial centroid indices: [1 5 9]

Initial centroid:
  Cluster 0: [-0.92870533 -1.01782672]
  Cluster 1: [1.78945661 0.25445668]
  Cluster 2: [-0.36242159 -4.8346769 ]

Iteration 1:
  Cluster sizes: [4 4 2]
New centroid positions:
  Cluster 0: [-0.53230671 -0.38168502]
  Cluster 1: [0.67104623 2.48095262]
  Cluster 2: [-0.27747903 -4.1985352 ]

Iteration 2:
  Cluster sizes: [4 3 3]
New centroid positions:
  Cluster 0: [ 0.44453273 -0.06361417]
  Cluster 1: [-0.62668734  3.6472124 ]
  Cluster 2: [ 0.03397702 -3.5623935 ]

Iteration 3:
  Cluster sizes: [5 2 3]
New centroid positions:
  Cluster 0: [0.15855945 0.25445668]
  Cluster 1: [-0.44736415  4.70744856]
  Cluster 2: [ 0.03397702 -3.5623935 ]

Iteration 4:
  Cluster sizes: [5 2 3]
New centroid positions:
  Cluster 0: [0.1585